In [1]:
from data.cleaning import read_csv
from core import Config
import pandas as pd

config = Config()

static_df: pd.DataFrame = read_csv(
    config.eda_filtered_dir,
    "eda_filtered_static_combined_2026.csv",
    "eda_filtered_static_dtypes_2026.csv"
)

historic_df: pd.DataFrame = read_csv(
    config.eda_filtered_dir,
    "eda_filtered_historic_2026.csv",
    "eda_filtered_historic_dtypes_2026.csv",
    [0, 1]
)

# Impute static dataframe

In [4]:
from data.constants.constants_hq import HQ

'''
Example how one loop of fill_na_by_modes looks like:

modes: dict[str, str] = {}
for group in filtered_df.groupby("TR.HQCountryCode")['Currency Code']:
    country = group[0]
    currency = group[1].mode().iloc[0]
    modes[country] = currency
mask = filtered_df['Currency Code'].isna()
filtered_df.loc[mask, 'Currency Code'] = filtered_df.loc[mask, 'TR.HQCountryCode'].map(modes)
'''

def fill_na_by_modes(df: pd.DataFrame) -> pd.DataFrame:
    fill_key_by_modes_of_value: dict[str, str] = {
        'Currency Code': 'TR.HQCountryCode',
        'TR.AssetCategory': 'TR.GICSSectorCode',
        'TR.BusinessSector': 'TR.GICSSectorCode',
        'TR.BusinessSectorScheme': 'TR.GICSSectorCode',
        'TR.CompanyParentType': 'TR.GICSSectorCode',
        'TR.HeadquartersRegionAlt': 'TR.HQCountryCode',
        'TR.InstrumentType': 'TR.GICSSectorCode',
        'TR.OrganizationType': 'TR.GICSSectorCode',
        'TR.PriceMainIndex': 'TR.HQCountryCode',
        'TR.RelatedOrgISO2': 'TR.HQCountryCode',
        'TR.RelatedOrgType': 'TR.GICSSectorCode',
    }
    modes: dict[str, str]
    mask: pd.Series

    for missing_value_col, col in fill_key_by_modes_of_value.items():
        modes = {}

        groups = df.groupby(col)[missing_value_col]
        for group in groups:
            modes[group[0]] = group[1].mode().iloc[0]

        mask = df[missing_value_col].isna()
        df.loc[mask, missing_value_col] = df.loc[mask, col].map(modes)

    return df

def fill_na_by_median(df: pd.DataFrame) -> pd.DataFrame:
    medians: dict[str, int] = {}

    for group in df.groupby('TR.GICSSectorCode')['Total Share Float']:
        total_shares = group[1].dropna()
        medians[group[0]] = int(total_shares.median())

    mask = df['Total Share Float'].isna()
    df.loc[mask, 'Total Share Float'] = df.loc[mask, 'TR.GICSSectorCode'].map(medians)

    return df

static_df = fill_na_by_modes(static_df)
static_df = fill_na_by_median(static_df)
static_df['TR.HeadquartersCity'] = static_df['TR.HeadquartersCity'].fillna(HQ)